# Automated Grape Leaf Disease Analysis: End-to-End Pipeline

This notebook provides a complete automated pipeline for:
1. **Dataset Generation**: Processing raw leaf images to create a segmented (background-removed) dataset.
2. **Classification**: Training an ensemble model (DenseNet + EfficientNet) on the segmented images.
3. **Severity Calculation**: Measuring the percentage of infection on each leaf.
4. **Visualization**: Displaying intermediate results (Masking, Background Removal, Disease Segmentation, and Diagnosis).

In [ ]:
# !pip install torch torchvision torchaudio scikit-learn matplotlib opencv-python pillow

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_curve, auc, confusion_matrix
from PIL import Image
import copy
import shutil
from tqdm.auto import tqdm

# Optional: Mount Google Drive if using Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
except:
    print("Not running in Google Colab or Drive mounting failed.")

## 1. Segmentation Module

### 1.1 Automated Dataset Pre-processing (Traditional Method)
This step takes your raw images and creates a new folder in your Drive containing only the isolated leaves. This is the robust, unsupervised method used for the current pipeline.

In [ ]:
# PATH CONFIGURATION
RAW_DATA_PATH = 'path_to_raw_dataset' # Path to folder with 4 subfolders
SEGMENTED_DATA_PATH = '/content/segmented_grape_dataset' # Local or Drive path for output

def isolate_leaf(image_path):
    image = cv2.imread(image_path)
    if image is None: return None, None
    
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Heuristic for green leaf segmentation
    lower_green = np.array([20, 30, 30])
    upper_green = np.array([100, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    
    # Morphological cleaning
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    leaf_only = cv2.bitwise_and(image_rgb, image_rgb, mask=mask)
    return leaf_only, mask

def create_segmented_dataset(src_root, dest_root):
    if os.path.exists(dest_root): shutil.rmtree(dest_root)
    os.makedirs(dest_root, exist_ok=True)
    
    categories = [d for d in os.listdir(src_root) if os.path.isdir(os.path.join(src_root, d))]
    
    for cat in categories:
        print(f"Processing category: {cat}")
        src_dir = os.path.join(src_root, cat)
        dest_dir = os.path.join(dest_root, cat)
        os.makedirs(dest_dir, exist_ok=True)
        
        images = [f for f in os.listdir(src_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for img_name in tqdm(images):
            src_path = os.path.join(src_dir, img_name)
            dest_path = os.path.join(dest_dir, img_name)
            
            segmented_leaf, _ = isolate_leaf(src_path)
            if segmented_leaf is not None:
                Image.fromarray(segmented_leaf).save(dest_path)

# UNCOMMENT TO RUN SEGMENTATION
# create_segmented_dataset(RAW_DATA_PATH, SEGMENTED_DATA_PATH)

### 1.2 Advanced Segmentation Models (Deep Learning)
As per your previous research, these architectures (U-Net, DeepLabV3+, FCN-8s) are included for future training. Note: These require ground-truth segmentation masks to be trained.

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(SimpleUNet, self).__init__()
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.ReLU(inplace=True)
            )
        self.enc1 = conv_block(in_channels, 64)
        self.enc2 = conv_block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = conv_block(128, 64)
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        u1 = self.up1(e2)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        return torch.sigmoid(self.final(d1))

def get_deeplabv3_plus(num_classes=1):
    # Using ResNet backbone as a proxy for DeepLabV3+
    model = models.segmentation.deeplabv3_resnet50(pretrained=True)
    model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
    return model

def get_fcn_8s(num_classes=1):
    model = models.segmentation.fcn_resnet50(pretrained=True)
    model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
    return model

## 2. Classification Module (Ensemble)

We use **DenseNet121** and **EfficientNet_B0** backbones for classification.

In [ ]:
def get_densenet_baseline(num_classes):
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model

def get_efficientnet_baseline(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

class SoftVotingEnsemble(nn.Module):
    def __init__(self, modelA, modelB):
        super().__init__()
        self.modelA = modelA
        self.modelB = modelB
    def forward(self, x):
        # Averaging logits for soft voting
        return (self.modelA(x) + self.modelB(x)) / 2

## 3. Training and Performance Evaluation

In [ ]:
def train_model(model, loaders, criterion, optimizer, num_epochs=10, device='cuda'):
    model = model.to(device)
    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss, running_corrects = 0.0, 0
            
            for inputs, labels in loaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
            epoch_acc = running_corrects.double() / len(loaders[phase].dataset)
            if phase == 'val':
                print(f'Val Acc: {epoch_acc:.4f}')
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_wts = copy.deepcopy(model.state_dict())
    
    model.load_state_dict(best_wts)
    return model

def evaluate_model(model, loader, class_names, device='cuda'):
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())
            
    return np.array(y_true), np.array(y_pred), np.array(y_probs)

def plot_roc_comparison(results_dict, class_names):
    plt.figure(figsize=(10, 8))
    for name, (y_true, y_probs) in results_dict.items():
        # Micro-average ROC
        fpr, tpr, _ = roc_curve(np.eye(len(class_names))[y_true].ravel(), y_probs.ravel())
        plt.plot(fpr, tpr, label=f'{name} (AUC = {auc(fpr, tpr):.2f})')
        
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve Comparison')
    plt.legend()
    plt.show()

## 4. Severity Calculation and Visualization

In [ ]:
def calculate_severity(leaf_rgb, mask):
    # Disease segmentation using Lab 'a' channel
    lab = cv2.cvtColor(leaf_rgb, cv2.COLOR_RGB2Lab)
    _, a, _ = cv2.split(lab)
    a_blurred = cv2.GaussianBlur(a, (5, 5), 0)
    _, disease_mask = cv2.threshold(a_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    disease_mask = cv2.bitwise_and(disease_mask, disease_mask, mask=mask)
    
    leaf_area = np.sum(mask > 0)
    disease_area = np.sum(disease_mask > 0)
    severity = (disease_area / leaf_area) * 100 if leaf_area > 0 else 0
    return disease_mask, severity

def visualize_pipeline(image_path, model, class_names, device='cuda'):
    orig_img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    leaf_only, mask = isolate_leaf(image_path)
    disease_mask, severity = calculate_severity(leaf_only, mask)
    
    # Classification
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    input_tensor = transform(leaf_only).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        _, pred = torch.max(output, 1)
        label = class_names[pred[0]]
    
    # Plots
    fig, ax = plt.subplots(1, 5, figsize=(25, 5))
    ax[0].imshow(orig_img); ax[0].set_title("1. Original Image")
    ax[1].imshow(mask, cmap='gray'); ax[1].set_title("2. Binary Mask (Leaf)")
    ax[2].imshow(leaf_only); ax[2].set_title("3. Background Removed")
    ax[3].imshow(disease_mask, cmap='hot'); ax[3].set_title("4. Disease Segmentation")
    
    res_img = leaf_only.copy()
    res_img[disease_mask > 0] = [255, 0, 0]
    ax[4].imshow(res_img)
    ax[4].set_title(f"5. Final: {label}\nSeverity: {severity:.2f}%")
    plt.show()

## 5. Main Execution Loop

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. SEGMENT DATASET (Run once)
    print("Starting dataset segmentation...")
    create_segmented_dataset(RAW_DATA_PATH, SEGMENTED_DATA_PATH)
    
    # 2. LOADERS
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    full_ds = datasets.ImageFolder(SEGMENTED_DATA_PATH, transform=transform)
    train_idx, temp_idx = train_test_split(np.arange(len(full_ds)), test_size=0.3, stratify=full_ds.targets, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=[full_ds.targets[i] for i in temp_idx], random_state=42)
    
    loaders = {
        'train': DataLoader(Subset(full_ds, train_idx), batch_size=32, shuffle=True),
        'val': DataLoader(Subset(full_ds, val_idx), batch_size=32, shuffle=False),
        'test': DataLoader(Subset(full_ds, test_idx), batch_size=32, shuffle=False)
    }
    num_classes = len(full_ds.classes)
    
    # 3. TRAIN MODELS
    print("\n--- Training Baseline 1: DenseNet121 ---")
    densenet = get_densenet_baseline(num_classes)
    densenet = train_model(densenet, loaders, nn.CrossEntropyLoss(), optim.AdamW(densenet.parameters(), lr=0.001), device=device)
    
    print("\n--- Training Baseline 2: EfficientNet_B0 ---")
    efficientnet = get_efficientnet_baseline(num_classes)
    efficientnet = train_model(efficientnet, loaders, nn.CrossEntropyLoss(), optim.AdamW(efficientnet.parameters(), lr=0.001), device=device)
    
    print("\n--- Evaluating Ensemble (Soft Voting) ---")
    ensemble = SoftVotingEnsemble(densenet, efficientnet)
    
    # 4. EVALUATION & COMPARISON
    results = {}
    for name, model in [('DenseNet121', densenet), ('EfficientNet_B0', efficientnet), ('Ensemble', ensemble)]:
        y_true, y_pred, y_probs = evaluate_model(model, loaders['test'], full_ds.classes, device=device)
        results[name] = (y_true, y_probs)
        print(f"\n{name} Classification Report:")
        print(classification_report(y_true, y_pred, target_names=full_ds.classes))
    
    plot_roc_comparison(results, full_ds.classes)
    
    # 5. VISUALIZE EXAMPLE PIPELINE
    print("\n--- Pipeline Visualization Example ---")
    sample_path = full_ds.samples[0][0]
    visualize_pipeline(sample_path, ensemble, full_ds.classes, device=device)

if __name__ == "__main__":
    main()